# Phase 2 · Step 3 — Dense vector search / retrieval

Embeds a plain-English question with the same model used for the chunks,
and asks the `papermind_chunks` collection (populated in
`03_embed_upsert.ipynb`) for the closest matches. See
`learning/04_rag_pipeline/03_dense_search.md` for the why behind these
choices.

## Connect

Same client/constants as notebooks 02 and 03 — just connecting, not
creating or writing anything.

In [1]:
from pathlib import Path

from fastembed import TextEmbedding
from qdrant_client import QdrantClient

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
COLLECTION_NAME = "papermind_chunks"
STORAGE_PATH = Path("data/qdrant")

client = QdrantClient(path=str(STORAGE_PATH))
assert client.collection_exists(COLLECTION_NAME)

info = client.get_collection(COLLECTION_NAME)
print("Points in collection:", info.points_count)

Points in collection: 4


## Embed the question

Must use the exact same model as the chunks, or the comparison is
meaningless.

In [2]:
question = "How is citation accuracy measured in this paper?"

embedder = TextEmbedding(model_name=EMBEDDING_MODEL)
query_vector = next(embedder.embed([question])).tolist()

print(f"Query vector length: {len(query_vector)}")

Query vector length: 384


## Search

`query_points` (the current API, not the deprecated `search`) returns the
closest stored vectors to the query vector, ranked by similarity score.

In [3]:
response = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=2,
    with_payload=True,
)

for result in response.points:
    print(f"--- page {result.payload['page_label']} (score={result.score:.3f}) ---")
    print(result.payload["text"])
    print()

--- page 4 (score=0.693) ---
3. Results
Across a held-out set of research papers, grounded answers with page citations were rated
as more trustworthy than ungrounded answers by human reviewers, even when the two answers
conveyed the same information.

--- page 1 (score=0.657) ---
Abstract
This paper studies retrieval-augmented generation for question answering over research
papers. We show that combining dense retrieval with a re-ranking stage improves citation
accuracy compared to retrieval alone.



## Sanity check

The sample PDF's "3. Results" page is the one that actually talks about
citation accuracy being rated by reviewers — confirm it comes back as the
top (highest-scoring) result.

In [4]:
top_result = response.points[0]
print("Top result page:", top_result.payload["page_label"])
assert "3. Results" in top_result.payload["text"] or "citation" in top_result.payload["text"].lower()
print("Top result is relevant — search is working end to end.")

Top result page: 4
Top result is relevant — search is working end to end.


## Next

Phase 2 · Step 4 adds hybrid search — combining this dense (meaning-based)
search with keyword/sparse search, so exact terms (like a specific model
name) aren't missed just because their embedding isn't the closest match.